# Let's Build GPT — From-Scratch Template

This notebook is a **fill-in-the-blank exercise companion** to `gpt_dev.ipynb`
(Andrej Karpathy's companion notebook for the
["Let's build GPT: from scratch, in code, spelled out."](https://www.youtube.com/watch?v=kCc8FmEb1nY)
video, part of the *Neural Networks: Zero to Hero* series). It keeps every
architectural line of `gpt_dev.ipynb` intact — same variable names, same
class structure, same final hyperparameters — but blanks out the lines you're
meant to derive yourself and adds `assert` checkpoints so you know immediately
if a TODO is right, without needing to compare outputs by eye.

One structural difference from `gpt_dev.ipynb`: its final `Head` /
`MultiHeadAttention` / `FeedFoward` / `Block` / model classes only appear
bundled together in one big "full finished code" cell (its cell 37). This
template splits that single cell into the same incremental, individually
testable TODOs used in sections 7–8 below — the code is equivalent, just
staged for exercise purposes.

**How to use it side by side with `gpt_dev.ipynb`:**
1. Open both notebooks as separate tabs in Jupyter/VS Code and put them in a
   split view (in VS Code: drag the `gpt_dev.ipynb` tab to the right half of
   the window) — one pane for reference, one for your TODOs.
2. Each section below names the exact `gpt_dev.ipynb` cell(s) it corresponds
   to, so you can jump straight to Andrej's explanation/markdown for that
   step instead of scrubbing the video.
3. Work top to bottom — later sections depend on earlier ones.
4. Each TODO cell has a comment describing the expected inputs, outputs, and
   tensor shapes, followed by an `assert` checkpoint that fails loudly if
   your implementation is wrong.
5. Section 6 ("mathematical trick") is intentionally exploratory/toy code —
   it exists to build intuition before Section 7's real `Head` implementation.

### Reference map: this notebook → `gpt_dev.ipynb`

| This notebook | `gpt_dev.ipynb` cells | What it covers |
|---|---|---|
| §1 Data | 1–4 | download & inspect Tiny Shakespeare |
| §2 Tokenization | 5–7 | vocab, `encode`/`decode`, tensor-encode the dataset |
| §3 Train/val split & batching | 8–12 | `train_data`/`val_data`, `block_size`, `get_batch` |
| §4 Bigram baseline model | 13 | `BigramLanguageModel` (no attention at all) |
| §5 Training loop (baseline) | 14–16 | `AdamW`, training loop, sample from the untrained/trained bigram |
| §6 Mathematical trick in self-attention | 17–24 | toy averaging → matmul trick → softmax → self-attention v4 |
| §7 `Head`, scaling, `MultiHeadAttention`, `FeedForward`, `Block`, `LayerNorm` | 25–34 | attention notes, scaled-attention variance demo, `LayerNorm1d` demo — the *concepts* behind cell 37's classes |
| §8 Full `GPTLanguageModel` + real training | 37 | assembled model + 5000-step training run |

Rough correspondence to the video's timeline:
- §1–2: data loading & tokenization (~0:00–12:00)
- §3: batching (~12:00–32:00)
- §4–5: bigram baseline model + its training loop (~32:00–38:00)
- §6: the "mathematical trick in self-attention" (~38:00–58:00, includes t=368s)
- §7: a single self-attention `Head`, multi-head attention, feedforward,
  residual blocks, LayerNorm (~58:00–1:37:00)
- §8: assembling the full GPT model + training it for real (~1:37:00–end)


## 0. Setup

In [1]:
import typing_extensions

print(typing_extensions.__file__)
print("deprecated:", hasattr(typing_extensions, "deprecated"))

from typing_extensions import deprecated
print("deprecated import: OK")

import torch
print("PyTorch:", torch.__version__)

c:\Users\Fadil\anaconda3\lib\site-packages\typing_extensions.py
deprecated: True
deprecated import: OK
PyTorch: 2.5.0.dev20240825


In [2]:
!pip install -U typing_extensions


     ---------------------------------------- 45.6/45.6 kB 1.1 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2


In [8]:
import sys
print(sys.version)

import typing_extensions
print(typing_extensions.__version__)

import torch

3.10.9 | packaged by Anaconda, Inc. | (main, Mar  1 2023, 18:18:15) [MSC v.1916 64 bit (AMD64)]


AttributeError: module 'typing_extensions' has no attribute '__version__'

In [9]:
import sys
import typing_extensions
import importlib.metadata

print("Python:", sys.version)
print("typing_extensions version:", importlib.metadata.version("typing_extensions"))
print("typing_extensions path:", typing_extensions.__file__)

try:
    from typing_extensions import deprecated
    print("deprecated: AVAILABLE")
except ImportError as e:
    print("deprecated: MISSING")
    print("Error:", e)

import torch
print("PyTorch:", torch.__version__)

Python: 3.10.9 | packaged by Anaconda, Inc. | (main, Mar  1 2023, 18:18:15) [MSC v.1916 64 bit (AMD64)]
typing_extensions version: 4.16.0
typing_extensions path: c:\Users\Fadil\anaconda3\lib\site-packages\typing_extensions.py
deprecated: MISSING
Error: cannot import name 'deprecated' from 'typing_extensions' (c:\Users\Fadil\anaconda3\lib\site-packages\typing_extensions.py)


ImportError: cannot import name 'deprecated' from 'typing_extensions' (c:\Users\Fadil\anaconda3\lib\site-packages\typing_extensions.py)

In [10]:
import sys
print(sys.executable)

import typing_extensions
print(typing_extensions.__file__)
print(hasattr(typing_extensions, "deprecated"))

c:\Users\Fadil\anaconda3\python.exe
c:\Users\Fadil\anaconda3\lib\site-packages\typing_extensions.py
False


In [ ]:
import sys

print("typing_extensions loaded:", "typing_extensions" in sys.modules)

if "typing_extensions" in sys.modules:
    te = sys.modules["typing_extensions"]
    print("Loaded from:", te.__file__)
    print("Has deprecated:", hasattr(te, "deprecated"))

typing_extensions loaded: True
Loaded from: c:\Users\Fadil\anaconda3\lib\site-packages\typing_extensions.py
Has deprecated: False


In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


Using device: cpu


## 1. Data
*(`gpt_dev.ipynb` cells 1–4)*

We use the same dataset Karpathy uses: the Tiny Shakespeare corpus (~1MB of
Shakespeare concatenated into one text file). This part is not the learning
objective of the exercise, so it's provided for you.


In [3]:
# Download the dataset (only needs to run once)
!curl -s -o input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('length of dataset in characters:', len(text))
print(text[:500])


length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## 2. Tokenization
*(`gpt_dev.ipynb` cells 5–7)*

We're building a **character-level** tokenizer: every unique character in the
text becomes one token. This is the simplest possible tokenizer — real GPTs
use subword tokenizers (e.g. BPE / tiktoken), but character-level keeps the
focus on the Transformer architecture itself.

### TODO 2.1 — vocabulary
Find the sorted list of unique characters that appear in `text`, and compute
`vocab_size`.


In [6]:
# TODO 2.1: build the vocabulary
# chars: sorted list of unique characters appearing in `text`
# vocab_size: how many unique characters there are

chars = sorted(set(text))      # TODO
vocab_size = len(chars)  # TODO

# --- checkpoint ---
assert chars is not None and vocab_size is not None, "fill in chars/vocab_size"
print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### TODO 2.2 — encode / decode

Build a mapping from characters to integers (`stoi`) and back (`itos`), then
write `encode(s)` (string -> list[int]) and `decode(l)` (list[int] -> string).


In [7]:
# TODO 2.2: character <-> integer mappings
itos = {char : i for char, i in enumerate(chars)}  # TODO: dict mapping char -> int, using `chars`
stoi = {i : char for char, i in enumerate(chars)}  # TODO: dict mapping int -> char

def encode(s):
    # TODO: take a string, return a list of ints 
    return [stoi[c] for c in s]

def decode(l):
    # TODO: take a list (or tensor) of ints, return a string
    result = ""
    for i in l:
        result += itos[i]
    return result

# --- checkpoint ---
test_str = "hii there"
assert decode(encode(test_str)) == test_str, "encode/decode round-trip failed"
print(encode(test_str))
print(decode(encode(test_str)))


[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


### TODO 2.3 — encode the full dataset

Encode the entire `text` into a single `torch.tensor` of dtype `torch.long`.


In [8]:
# TODO 2.3: encode the whole dataset into a tensor of shape (len(text),), dtype=torch.long
data = torch.tensor(encode(text), dtype=torch.long)  # TODO

# --- checkpoint ---
assert data is not None and data.dtype == torch.long
print(data.shape, data.dtype)
print(data[:100])


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


## 3. Train/val split and batching
*(`gpt_dev.ipynb` cells 8–12)*

### TODO 3.1 — split
Split `data` into `train_data` (first 90%) and `val_data` (last 10%).


In [9]:
# TODO 3.1

SPLIT = 0.9
n = round(len(data)*0.9)  # TODO: index marking the 90% split point
train_data = data[:n]  # TODO
val_data = data[n:]    # TODO

assert train_data is not None and val_data is not None
print(len(train_data), len(val_data))


1003855 111539


### `block_size` — context length

`block_size` (a.k.a. context length / `T`) is the maximum number of previous
tokens the model looks at when predicting the next one. A chunk of
`block_size + 1` tokens actually contains `block_size` training examples,
since every offset `t` gives you an (input, target) pair
`(chunk[:t+1], chunk[t+1])`.


In [10]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context.tolist()} the target is: {target}")


when input is [18] the target is: 47
when input is [18, 47] the target is: 56
when input is [18, 47, 56] the target is: 57
when input is [18, 47, 56, 57] the target is: 58
when input is [18, 47, 56, 57, 58] the target is: 1
when input is [18, 47, 56, 57, 58, 1] the target is: 15
when input is [18, 47, 56, 57, 58, 1, 15] the target is: 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] the target is: 58


### TODO 3.2 — `get_batch`

Write a function that samples a random minibatch of chunks from `train_data`
or `val_data`.

- Sample `batch_size` random starting indices `i` (valid range: so that
  `i + block_size` doesn't run off the end of the data).
- For each `i`, `x` is `data[i : i+block_size]` and `y` is the same window
  shifted by one: `data[i+1 : i+block_size+1]`.
- Stack them into tensors of shape `(batch_size, block_size)`.
- Move both to `device`.


In [11]:
import random

batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?

def get_batch(split):
    # TODO 3.2: implement minibatch sampling
    # split: 'train' or 'val'
    # returns: x, y  -- both of shape (batch_size, block_size)

    (data, start_i) = (train_data, 0) if split == 'train' else (val_data, len(train_data))

    i_list = [random.randint(start_i, start_i+len(data)-1) for _ in range(batch_size)]

    # print([data[i: i+block_size] for i in i_list])

    xb = torch.stack([data[i: i+block_size] for i in i_list])
    yb = torch.stack([data[i+1: i+block_size+1] for i in i_list])
    
    return xb, yb

# --- checkpoint ---
xb, yb = get_batch('train')
assert xb.shape == (batch_size, block_size)
assert yb.shape == (batch_size, block_size)
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)


inputs:
torch.Size([4, 8])
tensor([[ 1, 52, 53, 58, 12,  0,  0, 24],
        [ 5, 56, 58,  1, 58, 53, 53,  1],
        [ 1, 42, 43, 52, 12,  0,  0, 25],
        [ 1, 57, 46, 53, 59, 50, 42,  1]])
targets:
torch.Size([4, 8])
tensor([[52, 53, 58, 12,  0,  0, 24, 13],
        [56, 58,  1, 58, 53, 53,  1, 51],
        [42, 43, 52, 12,  0,  0, 25, 17],
        [57, 46, 53, 59, 50, 42,  1, 46]])


## 4. Baseline: Bigram Language Model
*(`gpt_dev.ipynb` cell 13)*

Before building attention, Karpathy first builds the simplest possible
neural language model: a bigram model. It predicts the next token using
*only* the current token (no context at all), via a single learnable
embedding table of shape `(vocab_size, vocab_size)` — row `i` holds the
logits for "what comes after token `i`".

### TODO 4.1 — `BigramLanguageModel`

- `__init__`: create `self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)`.
- `forward(idx, targets=None)`:
  - `idx` has shape `(B, T)`. Look it up in the embedding table to get
    `logits` of shape `(B, T, vocab_size)`.
  - If `targets` is not `None`, compute cross-entropy loss. **Careful:**
    `F.cross_entropy` expects logits of shape `(N, C)` and targets of shape
    `(N,)`, so you'll need to reshape/view `logits` and `targets` by folding
    `B` and `T` together before computing the loss.
  - Return `(logits, loss)` (loss is `None` if `targets is None`).
- `generate(idx, max_new_tokens)`:
  - `idx` is `(B, T)`, the current context.
  - Loop `max_new_tokens` times: get logits from `forward` (ignore loss),
    take only the **last time step** `logits[:, -1, :]` (shape `(B, C)`),
    softmax it into probabilities, sample one token per batch row with
    `torch.multinomial`, and append it to `idx` along the time dimension.
  - Return the extended `idx` of shape `(B, T + max_new_tokens)`.


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8678, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


(Output will be garbage — the model hasn't been trained yet. That's expected.)

## 5. Training loop (baseline)
*(`gpt_dev.ipynb` cells 14–16)*

### TODO 5.1
Standard PyTorch training loop: create an `AdamW` optimizer, then for
`max_iters` steps: sample a batch, run the model, zero gradients, backprop,
and step the optimizer.


In [16]:
# TODO 5.1: create the optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
max_iters = 5000
eval_interval = 500

for steps in range(max_iters):
    xb, yb = get_batch('train')

    # TODO 5.1: evaluate the loss, zero grads, backward, step
    # raise NotImplementedError
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % eval_interval == 0:
        print(f"step {steps}: loss {loss.item():.4f}")

# --- checkpoint ---
# a bigram model has no context at all, so it plateaus well short of a real
# LM -- but it should still clearly beat the ln(vocab_size) ~= 4.17 loss of
# an untrained/random model.
assert loss.item() < 2.7, f"expected the bigram model to train down below ~2.7, got {loss.item():.4f}"
print(loss.item())


step 0: loss 2.5646
step 500: loss 2.5872
step 1000: loss 2.5018
step 1500: loss 2.5238
step 2000: loss 2.5850
step 2500: loss 2.2365
step 3000: loss 2.5315
step 3500: loss 2.5564
step 4000: loss 2.4277
step 4500: loss 2.4038
2.510824680328369


In [18]:
# Generate from the trained bigram model
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=300)[0].tolist()))



NUn:
IS:
Yooure.
MEnsargooor w hillinde, ty iMPr thencasie mo mesolonlle ayoumo'd qule,
Anoora pbodea ERAREve andoumy tanomad llanoce idirart;
Anond blizeton ay w'athotongef ilf.
F ga' stheero athaciratiee.
ESCHA banbyoowhike o'd.

T:
Thiford, Topowins itsmy e halaig akine ICESor, foul ter gellerer 


## 6. The mathematical trick in self-attention
*(`gpt_dev.ipynb` cells 17–24)*

*(This is the section that starts around the video timestamp you linked,
t=368s / 6:08 — though the actual "trick" derivation happens a bit later,
around ~38 minutes in.)*

The goal: figure out how to let each token "communicate" with (a weighted
average of) the tokens **before** it in the sequence, in a way that is (a)
fast (no explicit for-loops) and (b) causal (a token can't see the future).

We'll build this up in three versions of increasing elegance, all computing
the *same* result, on a toy tensor.

First, a tiny warm-up (no TODO, just run it and inspect): a lower-triangular
matrix `a`, normalized so each row sums to 1, turns a matrix multiply `a @ b`
into a running average of `b`'s rows. This *is* the trick — everything else
in this section is applying it to `x`.


In [19]:
# warm-up demo -- no TODO, just run and inspect
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [20]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2  # batch, time, channels
x = torch.randn(B, T, C)
x.shape


torch.Size([4, 8, 2])

### TODO 6.1 — version 1: explicit loop

For every batch `b` and time step `t`, compute `xbow[b, t]` ("bag of
words") as the **average** of `x[b, 0:t+1]` (i.e. the mean over all
previous tokens including the current one).


In [31]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


### TODO 6.2 — version 2: matrix multiplication trick

The same running-average operation can be expressed as a single matrix
multiply per batch element: `xbow2 = wei @ x`, where `wei` is a
`(T, T)` matrix that is **lower-triangular** (so token `t` only attends to
tokens `<= t`) and each row **sums to 1** (so it's an average, not a sum).

Steps:
1. Build `tril = torch.tril(torch.ones(T, T))`.
2. Build `wei` from `tril` such that each row is uniform over the nonzero
   entries (i.e. row `t` has `1/(t+1)` in its first `t+1` columns).
3. Compute `xbow2 = wei @ x` — note `wei` is `(T, T)` and `x` is
   `(B, T, C)`; PyTorch will broadcast `wei` across the batch dimension.


In [32]:
# TODO 6.2: version 2 -- matrix multiply
tril = torch.tril(torch.ones(T, T))

wei = tril / tril.sum(1).view(-1, 1)  # TODO: normalize tril so each row sums to 1
xbow2 = wei @ x

# --- checkpoint ---
assert torch.allclose(xbow, xbow2, atol=1e-5), "version 1 and version 2 should match"

### TODO 6.3 — version 3: softmax

Same result again, but built the way it will generalize to real attention:
start from an all-zeros `(T, T)` affinity matrix `wei`, **mask** the upper
triangle with `-inf` using `tril == 0` (so those positions vanish after
softmax), then take `softmax` along the last dimension.


In [37]:
# TODO 6.3: version 3 -- softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
# TODO: wei = wei.masked_fill(tril == 0, float('-inf'))
# TODO: wei = F.softmax(wei, dim=-1)
wei = wei.masked_fill(tril == 0, float('-inf'))  # TODO
wei = F.softmax(wei, dim=-1)
xbow3 =  wei @ x # TODO: wei @ x

# --- checkpoint ---
assert torch.allclose(xbow, xbow3, atol=1e-5), "version 1 and version 3 should match"


**Why version 3 matters:** in real self-attention, `wei` won't start as
zeros — it will be computed from **data** (queries and keys), so different
tokens end up with different, *learned* affinities to their past, instead of
a uniform average. That's exactly what Section 7 builds.


## 7. A single self-attention `Head`
*(`gpt_dev.ipynb` cells 23–34, concepts assembled into classes in cell 37)*

Now replace the uniform averaging with real (scaled dot-product)
self-attention:

- Every token emits a **query** ("what am I looking for?") and a **key**
  ("what do I contain?"), each via a `Linear(C, head_size, bias=False)`
  layer.
- Affinity between token `t1` (query) and token `t2` (key) is their dot
  product: `wei = q @ k.transpose(-2, -1)`, giving shape `(B, T, T)`.
- **Scale** by `1/sqrt(head_size)` so the dot products (and thus the
  softmax) don't saturate for large `head_size`.
- **Mask** future positions with `-inf` (causal: token `t` can't attend to
  `t' > t`) using the same `tril` trick as version 3, then `softmax`.
- The output isn't `x` itself, but a third learned projection, the
  **value**: `v = value(x)`, and the head returns `wei @ v`.

### TODO 7.1 — implement `Head`


In [ ]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B, T, head_size)
q = query(x)  # (B, T, head_size)

# TODO 7.1a: compute attention weights
# wei = q @ k^T, scaled by 1/sqrt(head_size)  -> shape (B, T, T)
wei = None  # TODO

tril = torch.tril(torch.ones(T, T))
# TODO 7.1b: causal mask + softmax
# wei = wei.masked_fill(tril == 0, float('-inf'))
# wei = F.softmax(wei, dim=-1)

v = value(x)
out = None  # TODO 7.1c: wei @ v

# --- checkpoint ---
assert out is not None and out.shape == (B, T, head_size)
# these two checks hold regardless of the Linear layers' random init:
assert torch.allclose(wei.sum(dim=-1), torch.ones(B, T), atol=1e-5), "each row of wei should sum to 1 (it's a softmax)"
upper_triangle = torch.triu(torch.ones(T, T), diagonal=1).bool()
assert (wei[:, upper_triangle] == 0).all(), "future positions should be masked to exactly 0 after softmax"
print(out.shape)
print(wei[0])  # inspect: should be lower-triangular, rows summing to 1, but NOT uniform (unlike v2/v3)


### Aside — why scale by `1/sqrt(head_size)`?

*(No TODO here — run and inspect, matches `gpt_dev.ipynb` cells 26–31.)*

If `q` and `k` have unit variance, their raw dot product `q @ k^T` has
variance roughly `head_size` — it grows with `head_size`. Feeding
large-variance logits into `softmax` makes it "too peaky" (converges toward
one-hot), which kills gradient flow to all but one position. Dividing by
`sqrt(head_size)` keeps `wei` at roughly unit variance so softmax stays
diffuse.


In [ ]:
# aside -- no TODO, just run and inspect
k_demo = torch.randn(B, T, head_size)
q_demo = torch.randn(B, T, head_size)
wei_demo = q_demo @ k_demo.transpose(-2, -1) * head_size**-0.5
print('k.var() =', k_demo.var().item())
print('q.var() =', q_demo.var().item())
print('wei.var() =', wei_demo.var().item())  # should be close to 1, not head_size


In [ ]:
# softmax of "normal" logits stays fairly diffuse...
print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1))
# ...but the same logits scaled up (unscaled attention, effectively) go
# "too peaky" and converge toward one-hot:
print(torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]) * 8, dim=-1))


### TODO 7.2 — package it as an `nn.Module`

Turn the scratch code above into a reusable `Head` module. Add **dropout**
on the attention weights (regularizes which tokens get attended to), and
register `tril` as a buffer (not a parameter — it isn't learned, and
`register_buffer` makes sure it moves with `.to(device)`).


In [ ]:
n_embd = 32   # size of the token/position embeddings the head will receive as input
dropout = 0.0

class Head(nn.Module):
    """one head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TODO 7.2: same logic as 7.1, but:
        #  - infer B, T, C from x.shape
        #  - use self.tril[:T, :T] for the mask (so it works even if T < block_size)
        #  - apply self.dropout to wei before multiplying by v
        raise NotImplementedError

# --- checkpoint ---
h = Head(head_size=16)
test_in = torch.randn(4, block_size, n_embd)
test_out = h(test_in)
assert test_out.shape == (4, block_size, 16)
print(test_out.shape)


### TODO 7.3 — `MultiHeadAttention`

Instead of one attention head, run several **in parallel**, each with a
smaller `head_size` (typically `n_embd // num_heads`), then concatenate
their outputs along the channel dimension and pass through a final
`Linear` projection (+ dropout). This lets different heads specialize in
different kinds of relationships.


In [ ]:
class MultiHeadAttention(nn.Module):
    """multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)  # note: head_size * num_heads == n_embd
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TODO 7.3: run each head on x, concatenate along the last dim,
        # then project (self.proj) and apply dropout
        raise NotImplementedError

# --- checkpoint ---
mha = MultiHeadAttention(num_heads=4, head_size=n_embd // 4)
test_out = mha(test_in)
assert test_out.shape == (4, block_size, n_embd)
print(test_out.shape)


### TODO 7.4 — `FeedForward`

Attention lets tokens exchange information; the feedforward layer lets each
token "think" about what it gathered, independently per-position. Standard
Transformer choice: a 2-layer MLP that expands to `4 * n_embd` in the
hidden layer, with a `ReLU` in between, projecting back down to `n_embd`,
plus dropout at the end.


In [ ]:
class FeedForward(nn.Module):
    """a simple linear layer followed by a non-linearity"""

    def __init__(self, n_embd):
        super().__init__()
        # TODO 7.4: self.net = nn.Sequential(
        #     nn.Linear(n_embd, 4 * n_embd),
        #     nn.ReLU(),
        #     nn.Linear(4 * n_embd, n_embd),
        #     nn.Dropout(dropout),
        # )
        self.net = None  # TODO

    def forward(self, x):
        return self.net(x)

# --- checkpoint ---
ffwd = FeedForward(n_embd)
test_out = ffwd(test_in)
assert test_out.shape == (4, block_size, n_embd)


### TODO 7.5 — `Block`: communication + computation, with residuals and LayerNorm

A Transformer block interleaves:
1. Multi-head self-attention ("communication" between tokens)
2. Feedforward ("computation" per token)

Both are wrapped with:
- a **residual/skip connection** (`x = x + sublayer(x)`) — critical for
  gradient flow in deep networks, and
- **pre-norm LayerNorm** (`nn.LayerNorm(n_embd)` applied *before* the
  sublayer, i.e. `x = x + sublayer(ln(x))`), which is the modern variant
  Karpathy uses (differs slightly from the original "Attention Is All You
  Need" paper, which normalizes *after*).


In [ ]:
class Block(nn.Module):
    """Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # TODO 7.5: pre-norm residual connections
        # x = x + self.sa(self.ln1(x))
        # x = x + self.ffwd(self.ln2(x))
        raise NotImplementedError

# --- checkpoint ---
blk = Block(n_embd, n_head=4)
test_out = blk(test_in)
assert test_out.shape == (4, block_size, n_embd)


### Aside — what does `LayerNorm` actually normalize?

*(No TODO here — run and inspect, matches `gpt_dev.ipynb` cells 32–34. This is
the same `LayerNorm1d` class from the earlier
[makemore / batchnorm episode](../3/batch_norm.ipynb) in this series, renamed
from `BatchNorm1d`.)*

`BatchNorm` normalizes each *feature* across the batch (column-wise).
`LayerNorm` instead normalizes each *example* across its own features
(row-wise) — no batch statistics, no running mean/var, no train/eval mode
mismatch. That's why it's the standard choice for Transformers, where batch
statistics are awkward (variable sequence lengths, autoregressive
generation one token at a time, etc).


In [ ]:
# aside -- no TODO, just run and inspect
class LayerNorm1d:  # (used to be BatchNorm1d)

    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(1, keepdim=True)  # mean *per row* (per example)
        xvar = x.var(1, keepdim=True)    # var *per row*
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

torch.manual_seed(1337)
ln_demo = LayerNorm1d(100)
x_demo = torch.randn(32, 100)  # batch of 32, 100-dim features
x_demo = ln_demo(x_demo)
print(x_demo.shape)
print('one feature across the batch:', x_demo[:, 0].mean().item(), x_demo[:, 0].std().item())
print('one example across its features:', x_demo[0, :].mean().item(), x_demo[0, :].std().item())
# only the second line should be ~(0, 1) -- that's the row-wise normalization


## 8. Putting it all together: `GPTLanguageModel`
*(`gpt_dev.ipynb` cell 37, its "full finished code" cell)*

Now assemble the full model:

- `token_embedding_table`: `nn.Embedding(vocab_size, n_embd)` — what each
  token *is*.
- `position_embedding_table`: `nn.Embedding(block_size, n_embd)` — where in
  the sequence it *is* (since attention itself has no notion of order).
- A stack of `n_layer` `Block`s (`nn.Sequential`).
- A final `nn.LayerNorm(n_embd)` (`ln_f`).
- `lm_head`: `nn.Linear(n_embd, vocab_size)` mapping back to vocabulary
  logits.

### TODO 8.1 — hyperparameters

Fill in values for a CPU/Colab-friendly run. These are exactly
`gpt_dev.ipynb` cell 37's reference config (its "full finished code" cell)
— filling them in to match means your loss curve below should track the
one printed there (train/val loss ~1.7/1.8 after 5000 steps).

If you have a GPU, Karpathy's bigger config from the video is
`n_embd=384, n_head=6, n_layer=6, block_size=256, batch_size=64,
dropout=0.2` — worth trying as a stretch goal once the small version works.


In [ ]:
# TODO 8.1: hyperparameters -- match gpt_dev.ipynb cell 37's reference config
batch_size = None     # TODO: 16  -- how many independent sequences per batch?
block_size = None     # TODO: 32  -- max context length for predictions
max_iters = None       # TODO: 5000
eval_interval = None   # TODO: 100
learning_rate = 1e-3
eval_iters = 200
n_embd = None           # TODO: 64
n_head = None           # TODO: 4
n_layer = None          # TODO: 4
dropout = None          # TODO: 0.0


### TODO 8.2 — `estimate_loss`

A helper that estimates the average loss over `eval_iters` random batches,
for both `train` and `val` splits, with the model in `.eval()` mode (and
switched back to `.train()` afterward). Wrap it in `@torch.no_grad()` so it
doesn't build a computation graph.


In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    # TODO 8.2: for each split in ['train', 'val'], average the loss over
    # eval_iters batches and store it in out[split]
    raise NotImplementedError
    model.train()
    return out


### TODO 8.3 — `GPTLanguageModel`

- `__init__`: build the two embedding tables, the `nn.Sequential` of
  `Block`s, `ln_f`, and `lm_head`.
- `forward(idx, targets=None)`:
  - `idx` is `(B, T)`. Look up token embeddings `(B, T, n_embd)` and
    position embeddings `(T, n_embd)` (using
    `torch.arange(T, device=idx.device)`), and **add** them.
  - Pass through the blocks, then `ln_f`, then `lm_head` to get
    `logits` of shape `(B, T, vocab_size)`.
  - If `targets` given, compute cross-entropy loss (same reshape trick as
    the bigram model).
- `generate(idx, max_new_tokens)`:
  - Same idea as the bigram model's `generate`, **but** you must first
    crop `idx` to the last `block_size` tokens (`idx_cond = idx[:,
    -block_size:]`) before calling `forward`, since the position embedding
    table only has `block_size` rows.


In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # TODO 8.3a: token + position embeddings -> blocks -> ln_f -> lm_head
        # remember the cross-entropy reshape trick if targets is not None
        raise NotImplementedError

    def generate(self, idx, max_new_tokens):
        # TODO 8.3b: like the bigram generate(), but crop idx to the last
        # block_size tokens before each forward pass
        raise NotImplementedError

model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# --- checkpoint ---
# a freshly-initialized model should be about as confused as a uniform
# random guess over vocab_size options: loss ~= ln(vocab_size) ~= 4.17.
# this exercises forward(), the reshape trick, AND estimate_loss() at once.
losses = estimate_loss(m)
assert 3.5 < losses['train'].item() < 5.0, f"untrained loss looks wrong: {losses}"
print(losses)


### TODO 8.4 — train the full model

Same shape as the Section 5 training loop, but:
- use `estimate_loss` every `eval_interval` steps to report both train and
  val loss (watch for the val loss diverging from train loss — that's
  overfitting),
- move batches to `device`.


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(m)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # TODO 8.4: sample a batch, move to device, forward, backward, step
    raise NotImplementedError

# --- checkpoint ---
# with gpt_dev.ipynb's reference hyperparameters, Andrej's run lands at
# train/val loss ~1.66/1.82 after 5000 steps -- comfortably beating the
# bigram baseline's ~2.5-2.7 plateau. This threshold gives you headroom for
# a different random seed while still confirming attention is doing work.
final_losses = estimate_loss(m)
assert final_losses['val'].item() < 2.2, f"expected val loss well below the bigram baseline, got {final_losses}"
print(final_losses)
print(loss.item())


In [ ]:
# Generate from the fully trained GPT
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


## 9. Where to go next

- Compare the bigram model's generations (Section 5) to the full GPT's
  (Section 8) — the difference *is* the value of attention + depth.
- Try scaling `n_embd`, `n_layer`, `n_head`, `block_size` up (if you have a
  GPU) toward Karpathy's reference config and see how the sample quality
  changes.
- Re-watch the video for the sections you had to peek at solutions for —
  now that you've written the code, the intuition should land faster.
- Karpathy's own reference implementation (what this exercise is based on)
  lives in his `nanoGPT`/`ng-video-lecture` repos, if you want to check
  your work once you're done.
